In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
#import pyspark.sql.functions import *

spark = (
    SparkSession.builder
        .appName("dataframes_schemas")
        .master("local[*]")  
        #.master("spark://spark-master:7077")   
  
        .getOrCreate()
)

26/09/18 19:21:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
df_cliente = spark.read.csv('/opt/spark/storage/E-Commerce/customers.csv')
df_cliente.show(10)

+----------+----+-------+----------+---------------+
|       _c0| _c1|    _c2|       _c3|            _c4|
+----------+----+-------+----------+---------------+
|CustomerID| Age|   City|SignupDate|CustomerSegment|
|    100001|22.0| Tehran|2023-09-11|        Regular|
|    100002|55.0| Tabriz|2024-01-16|            VIP|
|    100003|49.0|  Karaj|2025-07-31|            New|
|    100004|39.0| Tehran|2023-04-23|            New|
|    100005|38.0| Tehran|2023-04-29|        Regular|
|    100006|59.0|Isfahan|2023-11-27|        Regular|
|    100007|22.0|  Karaj|2024-10-22|        Regular|
|    100008|51.0| Tabriz|2025-06-02|        Regular|
|    100009|27.0|  Karaj|2024-09-08|        Regular|
+----------+----+-------+----------+---------------+
only showing top 10 rows



In [3]:
df_cliente2 = spark.read.csv('/opt/spark/storage/E-Commerce/customers.csv',header=True)
df_cliente2.show(10)

+----------+----+-------+----------+---------------+
|CustomerID| Age|   City|SignupDate|CustomerSegment|
+----------+----+-------+----------+---------------+
|    100001|22.0| Tehran|2023-09-11|        Regular|
|    100002|55.0| Tabriz|2024-01-16|            VIP|
|    100003|49.0|  Karaj|2025-07-31|            New|
|    100004|39.0| Tehran|2023-04-23|            New|
|    100005|38.0| Tehran|2023-04-29|        Regular|
|    100006|59.0|Isfahan|2023-11-27|        Regular|
|    100007|22.0|  Karaj|2024-10-22|        Regular|
|    100008|51.0| Tabriz|2025-06-02|        Regular|
|    100009|27.0|  Karaj|2024-09-08|        Regular|
|    100010|NULL|Mashhad|2024-06-24|            New|
+----------+----+-------+----------+---------------+
only showing top 10 rows



In [4]:
df_cliente2.printSchema()

root
 |-- CustomerID: string (nullable = true)
 |-- Age: string (nullable = true)
 |-- City: string (nullable = true)
 |-- SignupDate: string (nullable = true)
 |-- CustomerSegment: string (nullable = true)



In [5]:
# insrindo o schema automaticamente
df_cliente_inferschema = spark.read.csv('/opt/spark/storage/E-Commerce/customers.csv',header=True,inferSchema=True)
df_cliente_inferschema.show(10)

+----------+----+-------+----------+---------------+
|CustomerID| Age|   City|SignupDate|CustomerSegment|
+----------+----+-------+----------+---------------+
|    100001|22.0| Tehran|2023-09-11|        Regular|
|    100002|55.0| Tabriz|2024-01-16|            VIP|
|    100003|49.0|  Karaj|2025-07-31|            New|
|    100004|39.0| Tehran|2023-04-23|            New|
|    100005|38.0| Tehran|2023-04-29|        Regular|
|    100006|59.0|Isfahan|2023-11-27|        Regular|
|    100007|22.0|  Karaj|2024-10-22|        Regular|
|    100008|51.0| Tabriz|2025-06-02|        Regular|
|    100009|27.0|  Karaj|2024-09-08|        Regular|
|    100010|NULL|Mashhad|2024-06-24|            New|
+----------+----+-------+----------+---------------+
only showing top 10 rows



In [6]:
df_cliente_inferschema.printSchema()

root
 |-- CustomerID: integer (nullable = true)
 |-- Age: double (nullable = true)
 |-- City: string (nullable = true)
 |-- SignupDate: date (nullable = true)
 |-- CustomerSegment: string (nullable = true)



In [8]:
#Map forma manual do schema(recomendado em ambiente de produção)

schema_map = """
    CustomerID int,
    Age DOUBLE,
    City STRING,
    SignupDate DATE,
    CustomerSegment STRING
"""


df_cliente_schema_manual = spark.read.csv(
    '/opt/spark/storage/E-Commerce/customers.csv',
    header=True,
    sep=',',
    schema=schema_map
)
df_cliente_schema_manual.show(10)

+----------+----+-------+----------+---------------+
|CustomerID| Age|   City|SignupDate|CustomerSegment|
+----------+----+-------+----------+---------------+
|    100001|22.0| Tehran|2023-09-11|        Regular|
|    100002|55.0| Tabriz|2024-01-16|            VIP|
|    100003|49.0|  Karaj|2025-07-31|            New|
|    100004|39.0| Tehran|2023-04-23|            New|
|    100005|38.0| Tehran|2023-04-29|        Regular|
|    100006|59.0|Isfahan|2023-11-27|        Regular|
|    100007|22.0|  Karaj|2024-10-22|        Regular|
|    100008|51.0| Tabriz|2025-06-02|        Regular|
|    100009|27.0|  Karaj|2024-09-08|        Regular|
|    100010|NULL|Mashhad|2024-06-24|            New|
+----------+----+-------+----------+---------------+
only showing top 10 rows



## structType<br>
Os tipos de dados usados:<br><br>

IntegerType(): números inteiros (ClienteID, Idade, RendaAnual, Frequencia)<br>
StringType(): texto (Genero, EstadoCivil, ProdutoPreferido)<br>
DateType(): datas (DataCadastro)<br><br>

Para que serve?<br>
Define o schema antes de carregar dados, garantindo:<br><br>

Validação de tipos: dados incompatíveis geram erro<br>
Performance: PySpark não precisa inferir tipos automaticamente<br>
Consistência: garante que os dados sempre tenham a estrutura esperada<br>

In [9]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, TimestampType
# from pyspark.sql.types import * importa tudo de Types

In [10]:
schema_struct = StructType([
    StructField("CustomerID", IntegerType(), True),
    StructField("Age", DoubleType(), True),
    StructField("City", StringType(), True),
    StructField("SignupDate", TimestampType(), True),
    StructField("CustomerSegment", StringType(), True)
])


df_cliente_schema_struct  = spark.read.csv(
    '/opt/spark/storage/E-Commerce/customers.csv',
    header=True,
    sep=',',
    schema=schema_struct
)
df_cliente_schema_struct.show(10)

+----------+----+-------+-------------------+---------------+
|CustomerID| Age|   City|         SignupDate|CustomerSegment|
+----------+----+-------+-------------------+---------------+
|    100001|22.0| Tehran|2023-09-11 00:00:00|        Regular|
|    100002|55.0| Tabriz|2024-01-16 00:00:00|            VIP|
|    100003|49.0|  Karaj|2025-07-31 00:00:00|            New|
|    100004|39.0| Tehran|2023-04-23 00:00:00|            New|
|    100005|38.0| Tehran|2023-04-29 00:00:00|        Regular|
|    100006|59.0|Isfahan|2023-11-27 00:00:00|        Regular|
|    100007|22.0|  Karaj|2024-10-22 00:00:00|        Regular|
|    100008|51.0| Tabriz|2025-06-02 00:00:00|        Regular|
|    100009|27.0|  Karaj|2024-09-08 00:00:00|        Regular|
|    100010|NULL|Mashhad|2024-06-24 00:00:00|            New|
+----------+----+-------+-------------------+---------------+
only showing top 10 rows



In [11]:
df_cliente_schema_struct.printSchema()

root
 |-- CustomerID: integer (nullable = true)
 |-- Age: double (nullable = true)
 |-- City: string (nullable = true)
 |-- SignupDate: timestamp (nullable = true)
 |-- CustomerSegment: string (nullable = true)



In [12]:
spark.stop()